In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from ydata_profiling import ProfileReport

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 50

data_folder = "../data/"
images_folder = "../data/images/"
df_total = pd.read_csv(os.path.join(data_folder, 'items_phase_1.csv'))
df_train = pd.read_csv(os.path.join(data_folder, 'items_train.csv'))
df_task_1 = pd.read_csv(os.path.join(data_folder, 'task_1.csv'))

# Notebook `items_phase_1.csv`

In [31]:
profile = ProfileReport(df_total, title="items_phase_1", explorative=True)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:01<00:00,  4.43it/s]



Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
df.sample(4)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo
176621,86776,53000.00,778,['1'],NaN,Körömcipők Baldowski,Baldowski Körömcipők D04888-4330-001 Bézs,hu
174801,1072778,17.99,231,['11'],NaN,Hugo Bodywear Nogavice/stopalke 3 pack,"Material: 87% bombaž, 12% poliamid, 1% elastan...",si
88192,743931,2699.00,"770,778",['1'],NaN,Lodičky Eva Minge,Eva Minge Lodičky ROSE-V1520-15 Červená,cz
79221,247834,60.00,230,['11'],NaN,Futbolo batai Puma,Futbolo batai Puma Future 7 Play Tt 107726 02 ...,lt


## Key takeaway
- missingy se musi resit u:
    - description (3%)
    - brandEditionTagId (99.8%) - to je asi target?
    - colorTagIdsString (3.1%)
---
# Notebook `task_1.csv`
- kazdy radek je jedna skupina se sloupci item - item4(to jsou id do items_phase_1.csv - itemId)
- Kazdy 

In [34]:
# profile = ProfileReport(df_task_1, title="task_1", explorative=True)
# profile.to_notebook_iframe()

In [33]:
df_task_1.head()

,item1,item2,item3,item4,item5
0,130622,292253,463442,483968,1253745
1,82627,388496,553738,638400,884327
2,46130,333489,644448,848154,1178149
3,150796,248537,742067,1206230,1280786
4,76610,196894,345145,620255,932761


---
# Spojeni datasetu


In [15]:
df_task_1.sample()

,item1,item2,item3,item4,item5
2839,273159,280445,349735,498672,693479


In [17]:
df.columns

Index(['itemId', 'price', 'colorTagIdsString', 'departmentIds',
       'brandEditionTagId', 'title', 'description', 'geo'],
      dtype='object')

In [18]:
df[df["itemId"] == 273159]

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo
95885,273159,9.0,6444,['1'],NaN,Olalook Dámska viacfarebná tkaná viskózová vzo...,"100% viskóza; Miery modelky: výška: 1,75 m, pr...",sk


In [24]:
df.geo.unique()

array(['gr', 'lv', 'hr', 'sk', 'hu', 'lt', 'bg', 'it', 'cz', 'si', 'ro',
       'ee', 'pl'], dtype=object)

---
# Dataset `items_train.csv`

In [37]:
print("Total records:", len(df_train))

Total records: 928234


In [41]:
# profile = ProfileReport(df_train, title="task_1", explorative=True)
# profile.to_notebook_iframe()


# Vycisteni dat 
- prevod na spolecnou menu 
- normalizace meny v danem geo uzemi
- doplneni null hodnot
- null ve sloupci `colorTagIdString` muzu nahradit 0 
- `colorTagIdString` a `departmentIds` je potreba roztrhnout - obsahuji vice hodnot oddelenych carkou

## Tvorba preprocessing pipeline

In [40]:
df_train

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo,label
0,692210,148.00,"230,232",['11'],NaN,Раница Rains 14480 Черен,NaN,bg,41599
1,943360,148.00,"230,232",['11'],NaN,Раница Rains,Rains Раница 14480 Черен,bg,41599
2,447879,179.90,230,['1'],NaN,Раница Rains Trail Cord Rolltop Backpack W3 в ...,Раница от колекция Rains. Моделът е направен о...,bg,41599
3,1169543,179.90,230,"['1', '11']",NaN,Раница Rains Trail Cord Rolltop Backpack W3 в ...,- Непромокаем модел.\n- Повишена водоустойчиво...,bg,41599
4,671883,1910.00,"230,232",['11'],NaN,Batoh Rains,Rains Batoh 14480 Černá,cz,41599
...,...,...,...,...,...,...,...,...,...
928229,480141,479.00,"775,1778",['1'],NaN,Top DHJ-TP-17733.22X ITALY MODA,Modelka má na sobě velikost jedna. Míry modelk...,cz,65996
928230,1278444,28.00,NaN,['42'],NaN,Adidas Detský ruksak Simpsonovci,Kupuj Detský ruksak Simpsonovci – hnedá na adi...,sk,214495
928231,439823,28.99,6444,['42'],NaN,"Detský ruksak adidas Performance čierna farba,...",- Polyester použitý na výrobu tohto modelu je ...,sk,214495
928232,34958,33.79,6444,['1'],NaN,Dámske šľapky Only,Dámske šľapky značky Only.\n- Model: Morgan-1\...,sk,140959


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from PriceGeoTransformer import PriceGeoTransformer
import numpy as np

imput_zero_cols = ['colorTagIdsString']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
])


# Define preprocessing for categorical features
categorical_features = ['geo']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Impute missing values with most frequent value
    ('onehot', OneHotEncoder(handle_unknown='ignore'))     # One-hot encode categorical features
])

# Combine preprocessing for numeric and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Create the complete pipeline with a classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])